In [ ]:
'''
Program ini dibuat untuk melakukan automatisasi pengolahan (cleaning) data text yang berguna untuk pemodelan model analisa sentimen.
=================================================
'''

'\n=================================================\nGraded Challenge 3\nNama  : [Risyadhana Syaifuddin]\nBatch : [HCK-036]\n\nProgram ini dirancang untuk melakukan penarikan data (scraping) produk seblak dari Tokopedia, \nmelakukan pembersihan data, analisis statistik untuk melihat potensi bisnis dropship, \ndan menyediakan antarmuka data sederhana menggunakan FastAPI.\n=================================================\n'

In [ ]:
#semua library yang dibutuhkan
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By #untuk nantinya bisa dilakukan scroll saat scraping
from selenium.webdriver.support.ui import WebDriverWait #untuk inisiasi Webdriver wait untuk time untuk auto scroll
from selenium.webdriver.support import expected_conditions as EC #untuk selector html tertuju ke kode cssnya/htmlnya
from scipy import stats
import time
import random #untuk jeda acak

In [ ]:
#inisiasi scrape_seblak dengan default perumpaman mendapatkan 350 data 
def scrape_seblak(target_data = 350):
    driver = webdriver.Chrome()
    all_products = []
    
    try:
        # Buka halaman awal
        url = "https://www.tokopedia.com/search?q=seblak&st=product"
        driver.get(url)
        
        #untuk memuat data dengan load_count
        load_count = 0
        while len(all_products) < target_data:
            print(f"Percobaan Muat Data ke-{load_count + 1}...")
            
            # Scroll perlahan ke bawah untuk memancing tombol muncul
            for _ in range(6): #tanpa perumpamaan dengan menggunakan "_"
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(random.uniform(0.8, 1.2))

            # Logika ketika melakukan Klik Tombol "Muat Lebih Banyak"
            try:
                # Menggunakan selector CSS berdasarkan class
                wait = WebDriverWait(driver, 7)
                load_more_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.css-1turmok-unf-btn")))
                
                # Scroll ke tombol agar terlihat oleh driver
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
                time.sleep(1)
                
                # Klik tombol
                load_more_btn.click()
                print(">>> Berhasil klik 'Muat Lebih Banyak'")
                
                load_count += 1
                time.sleep(random.uniform(3, 5)) # untuk menunggu loading konten baru
            except:
                print("Tombol tidak ditemukan atau sudah mencapai batas akhir.")
                driver.execute_script("window.scrollBy(0, 1500);")
                time.sleep(2)
                # kondisi Jika sudah cukup banyak tapi macet, saat dilakukan scraping
                if load_count > 5 and len(all_products) > 100: 
                    break
           
            
            html_content = driver.page_source    
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # Mencari container produk dengan memanggil element html
            items = soup.find_all('div', class_='css-5wh65g') 
            
            # Ekstraksi Data
            current_page_data = []
            
            for item in items:
                try:
                    # mendapatkan Nama Produk
                    findNama = item.find('div', {'data-testid': 'spnSRPProdName'})
                    nama = findNama.get_text(strip = True) if findNama else item.find('span', class_='+tnoqZhn89+NHUA43BpiJg==').get_text(strip=True)
                    
                    # Filter agar tidak mengambil produk komik/buku dengan bertemakan seblak
                    if any(x in nama.lower() for x in ['komik', 'buku', 'novel', 'manga']):
                        continue
                    
                    # mendapatkan Harga produk
                    findHarga = item.find('div', {'data-testid': 'spnSRPProdPrice'})
                    harga = findHarga.get_text(strip = True) if findHarga else item.find('div', class_='urMOIDHH7I0Iy1Dv2oFaNw== ').get_text(strip=True)
                    
                    # mendapatkan Penjual
                    findPenjual = item.find('span', class_='prd_link-shop-name')
                    if not findPenjual:
                        # Alternatif berdasarkan element HTML
                        findPenjual = item.find('span', class_='si3CNdiG8AR0EaXvf6bFbQ== gxi+fsEljOjqhjSKqjE+sw== flip')
                    penjual = findPenjual.get_text(strip=True) if findPenjual else "None"
                    
                    # Kota Penjual
                    kota = item.find('span', class_='prd_link-shop-loc')
                    if not kota:
                        # Mengambil span lokasi (biasanya span kedua di dalam info toko)
                        lokasi_elements = item.find_all('span', class_='gxi+fsEljOjqhjSKqjE+sw==')
                        kota = lokasi_elements[-1].get_text(strip=True) if lokasi_elements else "None"
                    else:
                        kota = kota.get_text(strip=True)

                    # Banyaknya Terjual (Handling Produk Baru)
                    findTerjual= item.find('span', class_='u6SfjDD2WiBlNW7zHmzRhQ==')
                    terjual = findTerjual.get_text(strip=True) if findTerjual else None
                    
                    # Rating Produk (Handling Produk Baru)
                    findRating= item.find('span', class_='_2NfJxPu4JC-55aCJ8bEsyw==')
                    rating = findRating.get_text(strip=True) if findRating else None

                    current_page_data.append({
                        "nama": nama,
                        "harga": harga,
                        "penjual": penjual,
                        "kota": kota,
                        "terjual": terjual,
                        "rating": rating
                    })
            
                except:
                    continue
                
            # Update list produk (menghindari duplikat saat append)
            all_products = current_page_data
            print(f"Data yang terdeteksi di layar: {len(all_products)}")
        
    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.quit()

    return pd.DataFrame(all_products)

# inisiasi finail dengan df_seblak
df_seblak = scrape_seblak()
df_seblak = df_seblak.drop_duplicates(subset=['nama', 'penjual'])#handle data yang duplicate berdasarkan kolom nama dan penjual

print(f"Total data bersih: {len(df_seblak)}")

df_seblak.to_csv('data_scraping.csv', index=False)
df_seblak.head()




Percobaan Muat Data ke-1...
>>> Berhasil klik 'Muat Lebih Banyak'
Data yang terdeteksi di layar: 237
Percobaan Muat Data ke-2...
>>> Berhasil klik 'Muat Lebih Banyak'
Data yang terdeteksi di layar: 398
Total data bersih: 384


,nama,harga,penjual,kota,terjual,rating
0,PROMO kerupuk Seblak mix seblak campur 1ball 8...,Rp34.550,Kurawa Snack.id,Kab. Bekasi,40+ terjual,5.0
1,1 BKS SEBLAK KEMPLANG KERUPUK JADOEL INSTAN FO...,Rp6.999,Rirra Store,Kab. Tangerang,1rb+ terjual,4.8
2,Hayang Seblak Instan Pedas Gurih Alami Sambal ...,Rp17.000,Hayang Seblak,Kab. Cirebon,1rb+ terjual,4.4
3,Kylafood Paket isi 5 Seblak Rempah Autentik,Rp70.000,Kylafood,Bandung,9rb+ terjual,4.9
4,terlaris Seblak Instan Garut 1 bungkus dengan ...,Rp2.999,SNACK OMA RR,Kab. Garut,50rb+ terjual,4.7


In [60]:
# A. Eksplorasi Awal
print(df_seblak.info())
print(df_seblak.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 384 entries, 0 to 397
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   nama     384 non-null    object
 1   harga    384 non-null    object
 2   penjual  384 non-null    object
 3   kota     384 non-null    object
 4   terjual  363 non-null    object
 5   rating   330 non-null    object
dtypes: object(6)
memory usage: 21.0+ KB
None
nama        0
harga       0
penjual     0
kota        0
terjual    21
rating     54
dtype: int64


#### Insight analisis :
    setelah melakukan web scraping, didapatkan beberapa kolom yang berisikan nama, harga, penjual, kota,terjual, dan rating. Selain itu, didapati jenis datatype setiap kolom, namun terdapat kata Rp dan . di kolom harga, dan rb dan + di kolom terjual oleh karena itu harus dilakukan data cleaning

In [61]:
df_clean = df_seblak.copy()

import pandas as pd
import re

def clean_terjual(x):
    # Jika data kosong atau None
    if pd.isna(x) or str(x).lower() == 'none' or str(x) == '':
        return 0
    
    # Ubah ke string dan kecilkan huruf
    s = str(x).lower()
    
    # Cek apakah mengandung 'rb' untuk pengali ribuan
    is_ribu = 'rb' in s
    
    # Ekstrak hanya angka dan tanda koma (untuk menangani 1,2 rb)
    # Menghapus 'terjual', '+', '.', dan spasi sekaligus
    s = re.sub(r'[^0-9,]', '', s)
    
    if s == '':
        return 0
    
    try:
        # Ganti koma menjadi titik agar bisa dikonversi ke float oleh Python
        s = s.replace(',', '.')
        angka = float(s)
        
        if is_ribu:
            return int(angka * 1000)
        else:
            return int(angka)
    except:
        return 0

# Pastikan mengambil dari dataframe asli hasil scraping (df_seblak)
# agar tidak mengambil data yang sudah terlanjur jadi 0
df_clean['terjual'] = df_seblak['terjual'].apply(clean_terjual)

# Cleaning kolom Harga (Menghilangkan Rp dan titik)
df_clean['harga'] = df_clean['harga'].str.replace('Rp', '', case=False).str.replace('.', '', regex=False)
df_clean['harga'] = pd.to_numeric(df_clean['harga'], errors='coerce').fillna(0).astype(int)

# Cleaning kolom Rating (Konversi ke float, fill missing value dengan 0)
df_clean['rating'] = pd.to_numeric(df_clean['rating'], errors='coerce').fillna(0.0)

# 3. Menyesuaikan tipe data akhir
print("\n--- Tipe Data Setelah Cleaning ---")
print(df_clean.dtypes)
display(df_clean.head())


--- Tipe Data Setelah Cleaning ---
nama        object
harga        int32
penjual     object
kota        object
terjual      int64
rating     float64
dtype: object


,nama,harga,penjual,kota,terjual,rating
0,PROMO kerupuk Seblak mix seblak campur 1ball 8...,34550,Kurawa Snack.id,Kab. Bekasi,40,5.0
1,1 BKS SEBLAK KEMPLANG KERUPUK JADOEL INSTAN FO...,6999,Rirra Store,Kab. Tangerang,1000,4.8
2,Hayang Seblak Instan Pedas Gurih Alami Sambal ...,17000,Hayang Seblak,Kab. Cirebon,1000,4.4
3,Kylafood Paket isi 5 Seblak Rempah Autentik,70000,Kylafood,Bandung,9000,4.9
4,terlaris Seblak Instan Garut 1 bungkus dengan ...,2999,SNACK OMA RR,Kab. Garut,50000,4.7


In [62]:
print(df_seblak['terjual'].unique()[:10])

['40+ terjual' '1rb+ terjual' '9rb+ terjual' '50rb+ terjual'
 '250+ terjual' '70+ terjual' '10rb+ terjual' '2 terjual' '80+ terjual'
 '20 terjual']



### C. Business Understanding (SMART Framework)
    Berdasarkan eksplorasi data di atas, kita dapat merumuskan tujuan bisnis sebagai berikut:

    Specific: Meningkatkan volume penjualan produk makanan seblak instan di platform Tokopedia dengan mengoptimalkan peningkatan performa rating, dan harga berkompetitif

    Measurable: Mencapai total penjualan (kolom terjual) minimal 500 unit per bulan untuk produk unggulan baru.

    Achievable: Menggunakan data rata-rata harga pasar untuk menentukan harga jual yang tepat, dan menguntungkan, namun tetap mengutamakan kualitas maupun kuantitas dari produk meskipun harganya murah

    Relevant: Pastinya meningkatkan penjualan dalam kategori makanan instan, pastinya akan mentrigger peningkatan visibilitas toko dalam setiap hasilpencarian Tokopedia.

    Time-bound: Target ini harus dicapai dalam jangka waktu minimal 4 bulan ke depan.

### D. Analysis: Menilik Potensi Pasar Seblak
Perjalanan analisis ini dimulai dengan membedah karakteristik dasar dari data yang telah dikumpulkan untuk memahami perilaku pasar seblak secara makro.

##### 1. Profil Statistik dan Karakteristik Distribusi Data

Bagaimana seblak diposisikan di pasar, dengan melakukan perhitungan statistik deskriptif pada tiga metrik utama: Harga, Banyaknya Terjual, dan Rating.

In [64]:
from scipy import stats

# Menghitung metrik untuk masing-masing kolom
metrics = df_clean[['harga', 'terjual', 'rating']].agg(['mean', 'median', 'std', 'skew', 'kurtosis']).T
display(metrics)

,mean,median,std,skew,kurtosis
harga,2.826987e+06,29999.5,5.105207e+07,19.558969,383.017589
terjual,9.452849e+03,500.0,4.029895e+04,9.849297,114.283675
rating,3.979167e+00,4.7,1.651974e+00,-1.915718,1.870470


##### Insight Analisis:

    Distribusi Harga: Data cenderung memiliki Skewness Positif (condong ke kiri), artinya sebagian besar penjual mengenakan harga rendah (ekonomis). Namun, disisi lain, terdapat beberapa barang "premium" dan barang pembelian massal dihargai jauh di atas rata-rata, menyebabkan rata-rata menyimpang dari median.

    Outlier yang Alami: Nilai Kurtosis yang tinggi pada kolom 'terjual' menunjukkan adanya indikasi outlier. Hal ini bisa terjadi, dikarenakan adanya pengaruh dari beberapa "penjual premium" yang mencapai ribuan unit terjual, sementara pendatang baru memulai dari nol. Ini menunjukkan bahwa pasar didominasi oleh satu pemenang.

    Kualitas Produk: Metrik Rating menunjukkan skewness negatif dengan rata-rata mendekati 4.7. Ini berarti ekspektasi konsumen terhadap seblak sangat tinggi, dan standar kualitas di pasar ini sudah proper.

##### 2. Estimasi Potensi Pendapatan Bulanan
Dengan mengasumsikan data terjual yaitu pastinya omzet bulanan yang didapat, dengan menghitung potensi pendapatan menggunakan confidence interval 95%.

In [ ]:
from scipy import stats
import numpy as np

# menghitung ulang omzet
df_clean['omzet'] = df_clean['harga'] * df_clean['terjual']

# menghitung statistik deskriptif (Mean, Median, Std, Skew, Kurtosis)
from scipy import stats
summary = df_clean[['harga', 'terjual', 'rating']].agg(['mean', 'median', 'std', 'skew', 'kurtosis']).T
print(summary)

# menghitung Confidence Interval 95% untuk Pendapatan (Omzet)
data_omzet = df_clean['omzet'][df_clean['omzet'] > 0] # Ambil yang ada penjualannya saja
mean_omzet = data_omzet.mean()
std_omzet = data_omzet.std()
n = len(data_omzet)

lower, upper = stats.t.interval(0.95, df=n-1, loc=mean_omzet, scale=std_omzet/(n**0.5))
print(f"\nPotensi Pendapatan/ Profit Bulanan: Rp{max(0, lower):,.2f} - Rp{upper:,.2f}")

                 mean   median           std       skew    kurtosis
harga    2.826987e+06  29999.5  5.105207e+07  19.558969  383.017589
terjual  9.452849e+03    500.0  4.029895e+04   9.849297  114.283675
rating   3.979167e+00      4.7  1.651974e+00  -1.915718    1.870470

Potensi Pendapatan Bulanan: Rp146,204,335.15 - Rp333,294,267.64


##### Insight Analisis:
    Dari hasil potensi pendapatan bulanan bisa dibilang mencerminkan variasi yang sangat besar di pasar seblak. Batas bawah sebesar Rp146 juta menunjukkan bahwa bahkan dalam estimasi yang konservatif, dikarenakan potensi pasar sangatlah besar dengan hasil potensi minimum dingka Rp.146 juta, dan batas potensi maksimum diangka Rp333 juta, mungkin harga ini melonjak diakibatkan adanya beberapa toko "top seller" yang didukung oleh volume penjualan sangat tinggi, dan proper

In [73]:
# 1. Menampilkan beberapa baris data
print("--- 5 Baris Data Teratas ---")
display(df_clean.head(10))

# 2. Menampilkan informasi rangkuman data
print("\n--- Informasi Rangkuman Data (Info) ---")
df_clean.info()

print("\n--- Statistik Deskriptif ---")
display(df_clean.describe())

# 3. Cek Missing Value
print("\n--- Cek Missing Value ---")
print(df_clean.isnull().sum())

--- 5 Baris Data Teratas ---


,nama,harga,penjual,kota,terjual,rating,omzet
0,PROMO kerupuk Seblak mix seblak campur 1ball 8...,34550,Kurawa Snack.id,Kab. Bekasi,40,5.0,1382000
1,1 BKS SEBLAK KEMPLANG KERUPUK JADOEL INSTAN FO...,6999,Rirra Store,Kab. Tangerang,1000,4.8,6999000
2,Hayang Seblak Instan Pedas Gurih Alami Sambal ...,17000,Hayang Seblak,Kab. Cirebon,1000,4.4,17000000
3,Kylafood Paket isi 5 Seblak Rempah Autentik,70000,Kylafood,Bandung,9000,4.9,630000000
4,terlaris Seblak Instan Garut 1 bungkus dengan ...,2999,SNACK OMA RR,Kab. Garut,50000,4.7,149950000
5,Paket Hemat Buy 1 Get 1 Kerupuk Seblak Mix Cam...,11000,Zia store16,Kab. Bandung,250,3.6,2750000
6,Seblak Kripca Seblak kripik Kaca Seblak Kering...,12000,sie jempol,Kab. Bandung,70,4.9,840000
7,SEBLAK KRUPUK CAMPUR/MIX SEBLAK BASRENG/BAWANG...,12500,KANGSEBLAK.ID,Kab. Bandung,10000,4.7,125000000
8,seblak kerupuk jengkol pedas daun jeruk Jengko...,9500,juragansnack100.id,Kab. Bandung,2,0.0,19000
9,Paket 10 Bungkus Seblak instant super komplit ...,36900,Raja Cemilan Garut,Kab. Garut,80,5.0,2952000



--- Informasi Rangkuman Data (Info) ---
<class 'pandas.core.frame.DataFrame'>
Index: 384 entries, 0 to 397
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   nama     384 non-null    object 
 1   harga    384 non-null    int32  
 2   penjual  384 non-null    object 
 3   kota     384 non-null    object 
 4   terjual  384 non-null    int64  
 5   rating   384 non-null    float64
 6   omzet    384 non-null    int64  
dtypes: float64(1), int32(1), int64(2), object(3)
memory usage: 22.5+ KB

--- Statistik Deskriptif ---


,harga,terjual,rating,omzet
count,3.840000e+02,384.000000,384.000000,3.840000e+02
mean,2.826987e+06,9452.848958,3.979167,2.266380e+08
std,5.105207e+07,40298.945998,1.651974,8.827909e+08
min,1.050000e+02,0.000000,0.000000,0.000000e+00
25%,1.500000e+04,67.500000,4.400000,1.517500e+06
50%,2.999950e+04,500.000000,4.700000,1.360000e+07
75%,4.500000e+04,5000.000000,4.800000,1.116765e+08
max,1.000000e+09,500000.000000,5.000000,1.050000e+10



--- Cek Missing Value ---
nama       0
harga      0
penjual    0
kota       0
terjual    0
rating     0
omzet      0
dtype: int64


In [74]:
print(df_clean['omzet'].describe())

count    3.840000e+02
mean     2.266380e+08
std      8.827909e+08
min      0.000000e+00
25%      1.517500e+06
50%      1.360000e+07
75%      1.116765e+08
max      1.050000e+10
Name: omzet, dtype: float64


##### 3. Komparasi Harga Berdasarkan Wilayah

In [71]:
#import stats
from scipy import stats

# Filter data
jabodetabek = ['jakarta', 'bogor', 'depok', 'tangerang', 'bekasi']
maskJabodetabek = df_clean['kota'].str.lower().str.contains('|'.join(jabodetabek))

hargaJabo = df_clean[maskJabodetabek]['harga']
hargaNonjabo = df_clean[~maskJabodetabek]['harga']#negasi buat ambil data non jabodetabek

# Uji T-Test Independen
t_stat, p_val_t = stats.ttest_ind(hargaJabo, hargaNonjabo, equal_var=False)

print(f"P-Value Uji Perbedaan Harga: {p_val_t:.4f}")

P-Value Uji Perbedaan Harga: 0.3003


##### Insight Analisis:
    Sebelum membahas hasil p-value yang didapat, mungkin banyak yang berasumsi bahwa penjual di jabodetabek harus memasang harga lebih mahal karena biaya operasional. Dengan indepent two sample t-test didapati bahwa p-value > 0.05, sehingga gagal menolak h0, sehingga membuktikan bahwa pasar digital sekelas Tokopedia, letak geografis atau wilayah tidak membuat harga berbeda secara signifikan. Sehingga, penjual di jabodetabek dapat memasang harga yang sama dengan penjual di daerah asal seblak yaitu yang didominasi di daerah seluruh jawa barat, tidak bisa dipungkiri untuk menjaga daya saing produk.

##### 4. Pembeli lebih cenderung suka dengan produk yang harganya murah?

In [72]:
from scipy import stats

# Menghitung korelasi Spearman dan P-Value
# pastikan df_clean sudah berisi data numerik pada kolom harga dan terjual
corr, p_value = stats.spearmanr(df_clean['harga'], df_clean['terjual'])

print(f"Koefisien Korelasi Spearman: {corr:.4f}")
print(f"P-Value: {p_value:.4e}")

Koefisien Korelasi Spearman: -0.2040
P-Value: 5.6287e-05


#### Insight Analisis ;
    Dari hasil analisa menggunakan korelasi Spearman bahwa p-value < 0.05, dikarenakan kecenderungan pembeli menyukai harga murah bukan terjadi karena suatu kebetulan, melainkan adalah suatu hal biasa yang ada di pasar sehingga membuktikan bahwa strategi penetapan harga rendah, bisa dibilang efektif dalam meningkatkan volume penjualan di platform tokopedia, mungkin bisa diimplemenatsikan juga ke platform yang lain seperti shopee, lazada, dan platform yang lainnya

### E. Conclusion

##### Berdasarkan seluruh rangkaian analisis data yang telah dilakukan terhadap produk seblak pada plaform tokopedia:

    Dari segi peluang pasar: pasar penjualan produk seblak instan melalui platform digital memiliki potensi ekonomi yang sangat besar. Dengan perkiraan penjualan bulanan sebesar Rp 146.204.335 hingga Rp 333.294.267, pasar ini ideal, dan worth it untuk pendatang baru dan ekspansi bisnis.

    Dari segi karakteristik produk: struktur harga pasar terdistribusi positif (positive skewness), dengan sebagian besar transaksi terkonsentrasi pada kisaran harga yang lebih rendah. Penjual didorong untuk menetapkan harga produk masing-masing dengan mendekati nilai rata-rata pasar untuk memastikan volume transaksi yang optimal.

    Dari segi persaingan regional: pengujian hipotesis mengungkapkan tidak ada perbedaan harga yang signifikan antara penjual di dalam dan di luar wilayah jabodetabek. Hal ini menunjukkan bahwa harga standar di platform tokopedia sedang berlangsung secara nasional, dengan efisiensi bahan baku dan biaya operasional, bukan perbedaan regional, sebagai pendorong utama yaitu menghasilkan keuntungan

    Dari segi strategi pemasaran: pengujian korelasi menunjukkan bahwa pembeli lebih menyukai harga rendah (korelasi negatif yang kuat). Namun, karena korelasi yang lemah, strategi terbaik adalah menawarkan "nilai uang" daripada hanya menawarkan harga terendah. Terutama befokus pada membangun kepercayaan melalui peringkat tinggi dan ulasan positif akan memiliki dampak jangka panjang yang lebih besar, daripada sekadar memangkas harga yang bisa dibilang diluar prediksi bmkg.